# Air Quality Prediction: AQI Regression and Category Classification

This notebook uses **two models only**:
- **Linear Regression** predicts the numerical `AQI`.
- **K-Nearest Neighbors (KNN) Classifier** predicts `AQI_Bucket`.

Place `air_quality_cleaned.csv` in the same folder as this notebook before running it from top to bottom. The `City` column is not used.

## 1. Import libraries and load the cleaned CSV

In [ ]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

df = pd.read_csv("air_quality_cleaned.csv")
print("Dataset shape:", df.shape)
display(df.head())
print("\nColumns:", df.columns.tolist())

## 2. Select input and output columns

In [ ]:
feature_columns = [
    "PM2.5", "PM10", "NO", "NO2", "NOx", "NH3", "CO",
    "SO2", "O3", "Benzene", "Toluene", "Xylene",
    "Year", "Month", "Day"
]
target_columns = ["AQI", "AQI_Bucket"]

# Date is converted to Year, Month, and Day below; City is intentionally excluded.
required_columns = ["Date", *[c for c in feature_columns if c not in ["Year", "Month", "Day"]], *target_columns]
missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"Required columns missing from air_quality_cleaned.csv: {missing_columns}")

df = df[required_columns].copy()
df.head()

## 3. Clean data and handle missing values

In [ ]:
# Remove rows without either target. Feature missing values are imputed inside each model pipeline.
df["AQI"] = pd.to_numeric(df["AQI"], errors="coerce")
df = df.dropna(subset=["AQI", "AQI_Bucket"]).copy()
df["AQI_Bucket"] = df["AQI_Bucket"].astype(str).str.strip()
df = df[df["AQI_Bucket"] != ""].copy()

# Ensure pollutant features are numeric; invalid values become missing and will be median-imputed.
pollutant_columns = [
    "PM2.5", "PM10", "NO", "NO2", "NOx", "NH3", "CO",
    "SO2", "O3", "Benzene", "Toluene", "Xylene"
]
for col in pollutant_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Rows after dropping missing targets:", len(df))
print("Missing input values (before imputation):")
# display(df[pollutant_columns].isna().sum().to_frame("missing_count"))

## 4. Convert Date into numerical features

In [ ]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.dropna(subset=["Date"]).copy()
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day

# Keep only the model inputs and the two targets. City and raw Date are not model features.
X = df[feature_columns].copy()
y_aqi = df["AQI"].copy()
y_category = df["AQI_Bucket"].copy()

print("Input feature columns:", X.columns.tolist())
print("AQI categories:", sorted(y_category.unique()))
display(X.head())

## 5. Split training and testing data

In [ ]:
# Both targets use exactly the same train/test observations.
# Stratify by category to preserve category proportions where possible.
try:
    X_train, X_test, y_aqi_train, y_aqi_test, y_category_train, y_category_test = train_test_split(
        X, y_aqi, y_category,
        test_size=0.2,
        random_state=42,
        stratify=y_category
    )
except ValueError:
    # Fallback for datasets with categories too rare to stratify.
    X_train, X_test, y_aqi_train, y_aqi_test, y_category_train, y_category_test = train_test_split(
        X, y_aqi, y_category,
        test_size=0.2,
        random_state=42
    )

print(f"Training rows: {len(X_train)} | Testing rows: {len(X_test)}")

## 6. Train Linear Regression to predict numerical AQI

In [ ]:
linear_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_aqi_train)
predicted_aqi = linear_model.predict(X_test)

## 7. Evaluate regression performance

In [ ]:
mae = mean_absolute_error(y_aqi_test, predicted_aqi)
rmse = np.sqrt(mean_squared_error(y_aqi_test, predicted_aqi))
r2 = r2_score(y_aqi_test, predicted_aqi)

print(f"MAE:  {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R²:   {r2:.3f}")

## 8. Train KNN Classifier to predict AQI category

In [ ]:
knn_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier(
        n_neighbors=5,
        weights="distance"
    ))
])

knn_model.fit(X_train, y_category_train)
predicted_category = knn_model.predict(X_test)

## 9. Evaluate classification performance

In [ ]:
accuracy = accuracy_score(y_category_test, predicted_category)
print(f"Accuracy: {accuracy:.3f}")
print("\nClassification report:")
print(classification_report(y_category_test, predicted_category, zero_division=0))

labels = sorted(y_category.unique())
cm = confusion_matrix(y_category_test, predicted_category, labels=labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
plt.title("KNN AQI Category Confusion Matrix")
plt.tight_layout()
plt.show()

## 10. Display actual vs. predicted results

In [ ]:
results = pd.DataFrame({
    "Actual AQI": y_aqi_test.to_numpy(),
    "Predicted AQI": predicted_aqi,
    "Actual Category": y_category_test.to_numpy(),
    "Predicted Category": predicted_category
}, index=X_test.index).sort_index()

display(results.head(20))

plt.figure(figsize=(7, 6))
plt.scatter(y_aqi_test, predicted_aqi, alpha=0.6)
min_val = min(float(y_aqi_test.min()), float(np.min(predicted_aqi)))
max_val = max(float(y_aqi_test.max()), float(np.max(predicted_aqi)))
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.xlabel("Actual AQI")
plt.ylabel("Predicted AQI")
plt.title("Linear Regression: Actual vs Predicted AQI")
plt.tight_layout()
plt.show()

## 11. Predict AQI and category for new pollutant measurements

In [ ]:
# Replace the example values with new pollutant measurements.
# Year, Month, and Day are also required because they are model features.
new_data = pd.DataFrame([{
    "PM2.5": 35.0,
    "PM10": 70.0,
    "NO": 5.0,
    "NO2": 20.0,
    "NOx": 25.0,
    "NH3": 10.0,
    "CO": 0.8,
    "SO2": 8.0,
    "O3": 30.0,
    "Benzene": 1.0,
    "Toluene": 5.0,
    "Xylene": 0.5,
    "Year": 2026,
    "Month": 10,
    "Day": 25
}], columns=feature_columns)

aqi_value = linear_model.predict(new_data)[0]
aqi_category = knn_model.predict(new_data)[0]

print(f"Predicted AQI: {aqi_value:.2f}")
print(f"AQI Category: {aqi_category}")
print("Note: the category is predicted directly by KNN, independently of the numerical AQI prediction.")

## 12. Save both trained models

In [ ]:
joblib.dump(linear_model, "linear_regression_aqi_model.pkl")
joblib.dump(knn_model, "knn_aqi_category_model.pkl")
print("Saved linear_regression_aqi_model.pkl")
print("Saved knn_aqi_category_model.pkl")